# Commutative Algebra Foundations v2

We develop the local commutative-algebra theory before transporting any construction through `Spec`. The primary objects are commutative rings, actual ring morphisms, quotient and localization universal arrows, local rings, fraction fields, $R$-algebras as a coslice category, free commutative algebras, and finite limits and colimits in $\mathbf{CRing}$.

In [1]:
from sage.all import *
from sage.categories.category import Category
from sage.categories.category_with_axiom import CategoryWithAxiom
from sage.categories.covariant_functorial_construction import (
    CovariantConstructionCategory,
    CovariantFunctorialConstruction,
)
from sage.categories.homset import Hom, Homset
from sage.categories.morphism import Morphism
from sage.categories.rings import Rings
from sage.categories.commutative_rings import CommutativeRings
from sage.categories.fields import Fields
from sage.categories.integral_domains import IntegralDomains
from sage.categories.algebras import Algebras
from sage.categories.commutative_algebras import CommutativeAlgebras
from sage.categories.map import FormalCompositeMap, Map
from sage.misc.cachefunc import cached_method
from sage.rings.morphism import RingHomomorphism
from sage.rings.polynomial.multi_polynomial_ring_base import MPolynomialRing_base
from sage.rings.polynomial.polynomial_ring import PolynomialRing_generic
from sage.structure.element import Element
from sage.structure.parent import Parent
from sage.structure.sage_object import SageObject

print('Sage version =', sage.version.version)
print('base category =', CommutativeRings())

Sage version = 10.10.beta0
base category = Category of commutative rings


## Arrows and commutative algebras over a base

The arrow category $\operatorname{Ar}(\mathbf{CRing})$ has ring morphisms as objects and commuting squares as morphisms. For a fixed commutative ring $R$, the coslice $(R\downarrow\mathbf{CRing})$ is the category of commutative $R$-algebras, including noncanonical structure morphisms.

In [2]:
def _native_equality_proves_equal(left, right):
    if left is right:
        return True
    try:
        return True if left == right else None
    except (TypeError, ValueError, AttributeError, NotImplementedError):
        return None


def _morphism_is_identity(morphism):
    if hasattr(morphism, 'is_identity'):
        try:
            return True if morphism.is_identity() else False
        except (TypeError, ValueError, AttributeError, NotImplementedError):
            return None
    return None


class MorphismEqualityCertificate(SageObject):
    def domain(self):
        raise NotImplementedError

    def verify(self, left, right):
        raise NotImplementedError


class PolynomialGeneratorEqualityCertificate(MorphismEqualityCertificate):
    def __init__(self, polynomial_ring):
        if not isinstance(
            polynomial_ring,
            (MPolynomialRing_base, PolynomialRing_generic),
        ):
            raise TypeError(
                'the generator certificate requires a polynomial-ring domain'
            )
        self._domain = polynomial_ring

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        if not all(
            left(generator) == right(generator)
            for generator in self._domain.gens()
        ):
            return False
        coefficient_ring = self._domain.base_ring()
        if coefficient_ring in (QQ, ZZ):
            return True
        coefficient_inclusion = self._domain.coerce_map_from(
            coefficient_ring
        )
        if coefficient_inclusion is None:
            return None
        left_on_coefficients = left * coefficient_inclusion
        right_on_coefficients = right * coefficient_inclusion
        if _native_equality_proves_equal(
            left_on_coefficients,
            right_on_coefficients,
        ) is True:
            return True
        if isinstance(
            coefficient_ring,
            (MPolynomialRing_base, PolynomialRing_generic),
        ):
            return PolynomialGeneratorEqualityCertificate(
                coefficient_ring
            ).verify(
                left_on_coefficients,
                right_on_coefficients,
            )
        return None


class ArrowCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'arrow_object'
    _functor_category = 'ArrowCategory'


class ArrowCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'ArrowCategory'
    _base_category_class = (Category,)

    def __init__(self, category):
        self._base_category = category
        self._args = tuple()
        self._morphism_equality_certificates = {}
        Category.__init__(self)

    def _repr_object_names(self):
        return (
            'arrows in '
            f'{self.base_category()._repr_object_names()}'
        )

    def arrow_category(self):
        return self

    def arrow_object(self, arrow):
        return ArrowObject(self, arrow)

    def register_morphism_equality_certificate(self, certificate):
        self._morphism_equality_certificates[
            id(certificate.domain())
        ] = certificate
        return self

    def morphisms_equal(self, left, right):
        if _native_equality_proves_equal(left, right) is True:
            return True
        if left.domain() is not right.domain():
            return False
        if left.codomain() is not right.codomain():
            return False
        if (
            _morphism_is_identity(left) is True
            and _morphism_is_identity(right) is True
        ):
            return True
        certificate = self._morphism_equality_certificates.get(
            id(left.domain())
        )
        if certificate is None:
            return None
        return certificate.verify(left, right)

    def square_status(
        self,
        source_arrow,
        target_arrow,
        source_leg,
        target_leg,
    ):
        try:
            left_composite = target_leg * source_arrow
            right_composite = target_arrow * source_leg
        except (TypeError, ValueError, AttributeError, NotImplementedError):
            return None
        return self.morphisms_equal(
            left_composite,
            right_composite,
        )


@cached_method
def _arrow_category(self):
    return ArrowCategoryConstruction(self)


Category.ArrowCategory = _arrow_category


class CosliceCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'coslice_object'
    _functor_category = 'CosliceCategory'

    def __init__(self, base_object):
        self._base_object = base_object

    def base_object(self):
        return self._base_object


class CosliceCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'CosliceCategory'
    _base_category_class = (Category,)

    def __init__(self, category, base_object):
        if base_object not in category:
            raise TypeError(
                'the fixed object is not an object of the base category'
            )
        self._base_category = category
        self._args = (base_object,)
        self._base_object = base_object
        Category.__init__(self)

    def base_object(self):
        return self._base_object

    def arrow_category(self):
        return self.base_category().ArrowCategory()

    def extra_super_categories(self):
        return [self.arrow_category()]

    def _repr_object_names(self):
        return (
            f'objects under {self._base_object} in '
            f'{self.base_category()._repr_object_names()}'
        )

    def coslice_object(self, structure_morphism):
        if structure_morphism.domain() is not self._base_object:
            raise ValueError(
                'a coslice object must have the fixed source'
            )
        return CosliceObject(self, structure_morphism)

    def initial_object(self):
        identity = self._base_object.Hom(
            self._base_object
        ).identity()
        return self.coslice_object(identity)

    def register_morphism_equality_certificate(self, certificate):
        self.arrow_category().register_morphism_equality_certificate(
            certificate
        )
        return self


@cached_method
def _coslice_category(self, base_object):
    return CosliceCategoryConstruction(self, base_object)


Category.CosliceCategory = _coslice_category


class ArrowObject(SageObject):
    _arrow_kind = 'arrow'

    def __init__(self, category, arrow):
        if not hasattr(arrow, 'domain') or not hasattr(arrow, 'codomain'):
            raise TypeError(
                'an arrow object requires a categorical morphism'
            )
        if arrow.domain() not in category.base_category():
            raise TypeError(
                'the arrow source is not in the base category'
            )
        if arrow.codomain() not in category.base_category():
            raise TypeError(
                'the arrow target is not in the base category'
            )
        self._arrow = arrow
        self._construction_category = category
        self._presentations = {}

    def category(self):
        return self._construction_category

    def construction_category(self):
        return self._construction_category

    def arrow_category(self):
        return self._construction_category.arrow_category()

    def arrow(self):
        return self._arrow

    def source(self):
        return self._arrow.domain()

    def target(self):
        return self._arrow.codomain()

    def arrow_kind(self):
        return self._arrow_kind

    def structure_morphism(self):
        return self._arrow

    def underlying_object(self):
        if self._arrow_kind == 'coslice':
            return self.target()
        raise AttributeError(
            'a general arrow has no distinguished underlying endpoint'
        )

    def register_presentation(self, presentation):
        self._presentations[presentation.kind] = presentation
        return self

    def presentation(self, kind):
        return self._presentations.get(kind)

    def _repr_(self):
        return f'Arrow object ({self._arrow})'

    def _Hom_(self, codomain, category=None):
        if self._arrow_kind == 'coslice':
            return CosliceHomset(self, codomain)
        return ArrowHomset(self, codomain)


class CosliceObject(ArrowObject):
    _arrow_kind = 'coslice'


class ArrowHomset(Homset):
    def __init__(self, domain, codomain):
        if domain.construction_category() is not (
            codomain.construction_category()
        ):
            raise TypeError(
                'arrow morphisms require a common construction category'
            )
        Homset.__init__(
            self,
            domain,
            codomain,
            category=domain.construction_category(),
        )

    def _element_constructor_(self, legs):
        if not isinstance(legs, (tuple, list)) or len(legs) != 2:
            raise TypeError(
                'an arrow morphism requires source and target legs'
            )
        return self.from_legs(legs[0], legs[1])

    def from_legs(self, source_leg, target_leg):
        return ArrowMorphism(
            self,
            source_leg,
            target_leg,
        )

    def identity(self):
        if self.domain() is not self.codomain():
            raise TypeError(
                'the identity is defined only on an endomorphism homset'
            )
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        target_identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        return self.from_legs(
            source_identity,
            target_identity,
        )


class CosliceHomset(ArrowHomset):
    def _element_constructor_(self, target_leg):
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        return self.from_legs(
            source_identity,
            target_leg,
        )

    def from_legs(self, source_leg, target_leg):
        identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        status = _morphism_is_identity(source_leg)
        if status is None:
            status = self.domain().arrow_category().morphisms_equal(
                source_leg,
                identity,
            )
        if status is False:
            raise ValueError(
                'a coslice morphism must have identity source leg'
            )
        if status is None:
            raise NotImplementedError(
                'the source leg could not be certified as the identity'
            )
        return ArrowMorphism(
            self,
            source_leg,
            target_leg,
        )


class ArrowMorphism(Element):
    def __init__(
        self,
        parent,
        source_leg,
        target_leg,
        square_verified=False,
    ):
        source_arrow = parent.domain()
        target_arrow = parent.codomain()
        if source_leg.domain() is not source_arrow.source():
            raise ValueError(
                'the source leg has the wrong domain'
            )
        if source_leg.codomain() is not target_arrow.source():
            raise ValueError(
                'the source leg has the wrong codomain'
            )
        if target_leg.domain() is not source_arrow.target():
            raise ValueError(
                'the target leg has the wrong domain'
            )
        if target_leg.codomain() is not target_arrow.target():
            raise ValueError(
                'the target leg has the wrong codomain'
            )
        if not square_verified:
            status = source_arrow.arrow_category().square_status(
                source_arrow.arrow(),
                target_arrow.arrow(),
                source_leg,
                target_leg,
            )
            if status is False:
                raise ValueError(
                    'the arrow square does not commute'
                )
            if status is None:
                raise NotImplementedError(
                    'the arrow square could not be certified'
                )
        self._source_leg = source_leg
        self._target_leg = target_leg
        Element.__init__(self, parent)

    def domain(self):
        return self.parent().domain()

    def codomain(self):
        return self.parent().codomain()

    def source_leg(self):
        return self._source_leg

    def target_leg(self):
        return self._target_leg

    def underlying_morphism(self):
        if self.domain().arrow_kind() == 'coslice':
            return self._target_leg
        raise AttributeError(
            'a general arrow morphism has two primary legs'
        )

    def is_identity(self):
        if self.domain() is not self.codomain():
            return False
        arrow_category = self.domain().arrow_category()
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        target_identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        source_status = arrow_category.morphisms_equal(
            self._source_leg,
            source_identity,
        )
        target_status = arrow_category.morphisms_equal(
            self._target_leg,
            target_identity,
        )
        if source_status is False or target_status is False:
            return False
        if source_status is True and target_status is True:
            return True
        raise NotImplementedError(
            'identity of the arrow morphism could not be certified'
        )

    def __mul__(self, right):
        if not isinstance(right, ArrowMorphism):
            return NotImplemented
        if right.codomain() is not self.domain():
            raise TypeError(
                'the arrow morphisms are not composable'
            )
        try:
            if right.is_identity():
                return self
        except NotImplementedError:
            pass
        try:
            if self.is_identity():
                return right
        except NotImplementedError:
            pass
        return ArrowMorphism(
            Hom(right.domain(), self.codomain()),
            self._source_leg * right._source_leg,
            self._target_leg * right._target_leg,
            square_verified=True,
        )


print('Installed arrow and coslice categories for commutative algebra.')

Installed arrow and coslice categories for commutative algebra.


## Quotients, localizations, and fraction fields as universal arrows

A quotient is the morphism $R\to R/I$. A localization is the morphism $R\to S^{-1}R$ together with its factorization property. For an integral domain, the fraction field is the localization at the multiplicative subset of nonzero elements. The codomain ring parent is retained, but the universal arrow is the primary object.

In [3]:
class MultiplicativeSubset(SageObject):
    def __init__(self, ring):
        if ring not in CommutativeRings():
            raise TypeError(
                'the ambient parent must be a commutative ring'
            )
        self._ring = ring

    def ring(self):
        return self._ring

    def contains(self, element):
        raise NotImplementedError


class FinitelyGeneratedMultiplicativeSubset(MultiplicativeSubset):
    def __init__(self, ring, generators):
        super().__init__(ring)
        self._generators = tuple(
            ring(generator)
            for generator in generators
        )
        if not self._generators:
            self._generators = (ring.one(),)
        if any(
            generator.is_zero()
            for generator in self._generators
        ):
            raise ValueError(
                'a multiplicative subset may not contain zero'
            )

    def generators(self):
        return self._generators

    def _repr_(self):
        return (
            f'Multiplicative subset of {self.ring()} '
            f'generated by {self._generators}'
        )


class ComplementOfPrimeIdeal(MultiplicativeSubset):
    def __init__(self, prime_ideal):
        ring = prime_ideal.ring()
        if not prime_ideal.is_prime():
            raise ValueError(
                'the ideal must be prime'
            )
        super().__init__(ring)
        self._prime_ideal = prime_ideal

    def prime_ideal(self):
        return self._prime_ideal

    def contains(self, element):
        return (
            self.ring()(element)
            not in self._prime_ideal
        )

    def _repr_(self):
        return (
            f'Complement of {self._prime_ideal} '
            f'in {self.ring()}'
        )


class NonzeroElements(MultiplicativeSubset):
    def __init__(self, domain):
        if domain not in IntegralDomains():
            raise TypeError(
                'the nonzero elements are multiplicative only in a domain'
            )
        super().__init__(domain)

    def contains(self, element):
        return not self.ring()(element).is_zero()

    def _repr_(self):
        return f'Nonzero elements of {self.ring()}'


class CanonicalLocalizationMorphism(RingHomomorphism):
    def __init__(self, ring, localization_ring):
        if localization_ring.base_ring() is not ring:
            raise ValueError(
                'the localization parent has the wrong base ring'
            )
        self._localization_ring = localization_ring
        RingHomomorphism.__init__(
            self,
            Hom(ring, localization_ring),
        )

    def _call_(self, element):
        return self._localization_ring(element)

    def _repr_defn(self):
        return 'Canonical localization morphism'


class LocalizationExtensionMorphism(RingHomomorphism):
    def __init__(
        self,
        localization_ring,
        target_ring,
        base_map,
        inverted_generators,
    ):
        if localization_ring.base_ring() is not base_map.domain():
            raise ValueError(
                'the base map has the wrong domain'
            )
        if target_ring is not base_map.codomain():
            raise ValueError(
                'the base map has the wrong codomain'
            )
        self._base_map = base_map
        self._inverted_generators = tuple(
            base_map.domain()(generator)
            for generator in inverted_generators
        )
        if not all(
            target_ring(base_map(generator)).is_unit()
            for generator in self._inverted_generators
        ):
            raise ValueError(
                'an inverted generator does not map to a unit'
            )
        RingHomomorphism.__init__(
            self,
            Hom(localization_ring, target_ring),
        )

    def base_map(self):
        return self._base_map

    def _call_(self, element):
        numerator_image = self._base_map(
            element.numerator()
        )
        denominator_image = self._base_map(
            element.denominator()
        )
        if not denominator_image.is_unit():
            raise ArithmeticError(
                'the denominator image is not a unit'
            )
        return (
            numerator_image
            * denominator_image.inverse_of_unit()
        )

    def _repr_defn(self):
        return (
            'Induced by the localization universal property'
        )


class LocalizationEqualityCertificate(MorphismEqualityCertificate):
    def __init__(self, localization_arrow, base_certificate):
        self._localization_arrow = localization_arrow
        self._domain = localization_arrow.codomain()
        self._base_certificate = base_certificate

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        return self._base_certificate.verify(
            left * self._localization_arrow,
            right * self._localization_arrow,
        )


class LocalizationUniversalProperty(SageObject):
    def __init__(self, multiplicative_subset, localization_arrow):
        self._multiplicative_subset = multiplicative_subset
        self._arrow = localization_arrow

    def multiplicative_subset(self):
        return self._multiplicative_subset

    def arrow(self):
        return self._arrow

    def localization_ring(self):
        return self._arrow.codomain()

    def factor(self, base_map):
        if base_map.domain() is not self._arrow.domain():
            raise ValueError(
                'the map has the wrong source ring'
            )
        subset = self._multiplicative_subset
        if not isinstance(
            subset,
            FinitelyGeneratedMultiplicativeSubset,
        ):
            raise NotImplementedError(
                'the current factorization backend requires a finitely generated multiplicative subset'
            )
        return LocalizationExtensionMorphism(
            self.localization_ring(),
            base_map.codomain(),
            base_map,
            subset.generators(),
        )


class FractionFieldEmbeddingMorphism(RingHomomorphism):
    def __init__(self, domain, fraction_field):
        if domain not in IntegralDomains():
            raise TypeError(
                'a fraction field requires an integral domain'
            )
        self._fraction_field = fraction_field
        RingHomomorphism.__init__(
            self,
            Hom(domain, fraction_field),
        )

    def _call_(self, element):
        return self._fraction_field(element)

    def _repr_defn(self):
        return 'Canonical fraction-field embedding'


class FractionFieldExtensionMorphism(RingHomomorphism):
    def __init__(
        self,
        source_domain,
        fraction_field,
        target_field,
        base_map,
    ):
        if source_domain is not base_map.domain():
            raise ValueError(
                'the base map has the wrong domain'
            )
        if target_field not in Fields():
            raise TypeError(
                'the target must be a field'
            )
        if target_field is not base_map.codomain():
            raise ValueError(
                'the base map has the wrong codomain'
            )
        self._base_map = base_map
        RingHomomorphism.__init__(
            self,
            Hom(fraction_field, target_field),
        )

    def _call_(self, element):
        numerator_image = self._base_map(
            element.numerator()
        )
        denominator_image = self._base_map(
            element.denominator()
        )
        if denominator_image.is_zero():
            raise ZeroDivisionError(
                'the base map kills a denominator'
            )
        return numerator_image / denominator_image

    def _repr_defn(self):
        return (
            'Induced by the fraction-field universal property'
        )


class InjectiveMapToFieldCertificate(SageObject):
    def __init__(self, base_map, theorem=None):
        if base_map.domain() not in IntegralDomains():
            raise TypeError(
                'the source must be an integral domain'
            )
        if base_map.codomain() not in Fields():
            raise TypeError(
                'the target must be a field'
            )
        verified = isinstance(
            base_map,
            FractionFieldEmbeddingMorphism,
        )
        if not verified and hasattr(
            base_map,
            'is_injective',
        ):
            try:
                verified = (
                    base_map.is_injective() is True
                )
            except (
                NotImplementedError,
                TypeError,
                AttributeError,
            ):
                verified = False
        if not verified:
            raise NotImplementedError(
                'injectivity of the map into the field could not be certified'
            )
        self._base_map = base_map
        self._theorem = (
            str(theorem)
            if theorem is not None
            else 'injectivity verified by the morphism implementation'
        )

    def base_map(self):
        return self._base_map

    def theorem(self):
        return self._theorem


class FractionFieldUniversalProperty(LocalizationUniversalProperty):
    def factor(self, base_map, certificate):
        if certificate.base_map() is not base_map:
            raise ValueError(
                'the injectivity certificate belongs to another map'
            )
        return FractionFieldExtensionMorphism(
            self.arrow().domain(),
            self.localization_ring(),
            base_map.codomain(),
            base_map,
        )


class QuotientEqualityCertificate(MorphismEqualityCertificate):
    def __init__(self, quotient_arrow, base_certificate):
        self._quotient_arrow = quotient_arrow
        self._domain = quotient_arrow.codomain()
        self._base_certificate = base_certificate

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        return self._base_certificate.verify(
            left * self._quotient_arrow,
            right * self._quotient_arrow,
        )


class QuotientUniversalProperty(SageObject):
    def __init__(self, ideal, quotient_arrow):
        if ideal.ring() is not quotient_arrow.domain():
            raise ValueError(
                'the ideal belongs to another ring'
            )
        self._ideal = ideal
        self._arrow = quotient_arrow

    def ideal(self):
        return self._ideal

    def arrow(self):
        return self._arrow

    def quotient_ring(self):
        return self._arrow.codomain()

    def factor(self, base_map):
        if base_map.domain() is not self._arrow.domain():
            raise ValueError(
                'the map has the wrong source ring'
            )
        if not all(
            base_map(generator).is_zero()
            for generator in self._ideal.gens()
        ):
            raise ValueError(
                'the map does not annihilate the quotient ideal'
            )
        quotient_ring = self.quotient_ring()
        source_ring = self._arrow.domain()
        if not hasattr(quotient_ring, 'gens'):
            raise NotImplementedError(
                'no finite-generator quotient factorization backend applies'
            )
        if len(quotient_ring.gens()) != len(source_ring.gens()):
            raise NotImplementedError(
                'the quotient generator presentation is incompatible with the source presentation'
            )
        generator_images = tuple(
            base_map(generator)
            for generator in source_ring.gens()
        )
        try:
            return quotient_ring.hom(
                generator_images,
                base_map.codomain(),
            )
        except Exception as error:
            raise NotImplementedError(
                'the native quotient homomorphism constructor rejected the finite presentation'
            ) from error


def _multiplicative_subset(self, generators):
    return FinitelyGeneratedMultiplicativeSubset(
        self,
        generators,
    )


def _localization_morphism(self, multiplicative_subset):
    if multiplicative_subset.ring() is not self:
        raise ValueError(
            'the multiplicative subset belongs to another ring'
        )
    if isinstance(
        multiplicative_subset,
        FinitelyGeneratedMultiplicativeSubset,
    ):
        localization_ring = self.localization(
            multiplicative_subset.generators()
        )
        arrow = CanonicalLocalizationMorphism(
            self,
            localization_ring,
        )
        return LocalizationUniversalProperty(
            multiplicative_subset,
            arrow,
        )
    if isinstance(
        multiplicative_subset,
        NonzeroElements,
    ):
        fraction_field = self.fraction_field()
        arrow = FractionFieldEmbeddingMorphism(
            self,
            fraction_field,
        )
        return FractionFieldUniversalProperty(
            multiplicative_subset,
            arrow,
        )
    if isinstance(
        multiplicative_subset,
        ComplementOfPrimeIdeal,
    ):
        return _OSCAR_PRIME_LOCALIZATION_BACKEND.localize(
            multiplicative_subset
        )
    raise NotImplementedError(
        'no localization backend applies to this multiplicative subset'
    )


def _fraction_field_morphism(self):
    return self.localization_morphism(
        NonzeroElements(self)
    )


def _localization_at_prime(self, prime_ideal):
    return self.localization_morphism(
        ComplementOfPrimeIdeal(prime_ideal)
    )


def _quotient_morphism(self, ideal, names=None):
    if ideal.ring() is not self:
        raise ValueError(
            'the ideal belongs to another ring'
        )
    quotient_ring = (
        self.quotient(ideal)
        if names is None
        else self.quotient(
            ideal,
            names=names,
        )
    )
    quotient_arrow = quotient_ring.cover()
    if quotient_arrow.domain() is not self:
        raise NotImplementedError(
            'the native quotient parent does not expose the required source-ring quotient morphism'
        )
    return QuotientUniversalProperty(
        ideal,
        quotient_arrow,
    )


_CommutativeRingParentMethods = (
    CommutativeRings().parent_class
)
for method_name, method in (
    ('multiplicative_subset', _multiplicative_subset),
    ('localization_morphism', _localization_morphism),
    ('fraction_field_morphism', _fraction_field_morphism),
    ('localization_at_prime', _localization_at_prime),
    ('quotient_morphism', _quotient_morphism),
):
    if method_name in _CommutativeRingParentMethods.__dict__:
        raise RuntimeError(
            f'the commutative-ring category already defines {method_name}'
        )
    setattr(
        _CommutativeRingParentMethods,
        method_name,
        method,
    )

print('Installed quotient, localization, and fraction-field universal arrows.')

Installed quotient, localization, and fraction-field universal arrows.


## Prime localizations through Oscar

The semantic operation remains localization of a commutative ring at the complement of a prime ideal. Oscar supplies the local commutative-algebra backend; Sage retains the parent, morphism, ideal, residue-field, and category interfaces.

The present executed backend covers $\mathbf Z_{(p)}$ and prime localizations of multivariate polynomial rings over $\mathbf Q$. Unsupported commutative-ring presentations remain explicit implementation boundaries of the same mathematical operation.

**Bridge note.** The installed Julia bridge reconstructs transferred multivariate polynomials in a new Oscar parent and cannot materialize `QQMPolyRingElem` values back into Sage. The bounded adapter below therefore transports coefficient--exponent data exactly while Oscar continues to own localization and ideal arithmetic.

In [4]:
from sage.categories.category_singleton import Category_singleton
from sage.rings.ideal import Ideal_generic
from sage.structure.element import RingElement
from sage.structure.richcmp import op_EQ, op_NE
from sage_julia_bridge import julia
from sage_julia_bridge.interface import JuliaHandle


class LocalRings(Category_singleton):
    """Commutative local rings, with membership certified per parent."""

    def super_categories(self):
        return [CommutativeRings()]

    def _repr_object_names(self):
        return 'local rings'


class OscarBridgeDriver(SageObject):
    """Persistent Oscar worker and exact finite-presentation transport."""

    def __init__(self, bridge=None):
        self._bridge = julia if bridge is None else bridge
        self._generation = None

    def bridge(self):
        return self._bridge

    def ensure_ready(self):
        generation = getattr(self._bridge, '_generation', None)
        if self._generation == generation:
            return self
        self._bridge.eval(r'''
begin
    using Oscar
    _ag_first(x) = x[1]
    _ag_second(x) = x[2]
    _ag_apply(f, x) = f(x)
    _ag_add(x, y) = x + y
    _ag_sub(x, y) = x - y
    _ag_mul(x, y) = x * y
    _ag_neg(x) = -x
    _ag_inv(x) = inv(x)
    _ag_eq(x, y) = x == y
    _ag_iszero(x) = iszero(x)
    _ag_isone(x) = isone(x)
    _ag_isunit(x) = is_unit(x)
    _ag_coerce(R, x) = R(x)
    _ag_numerator(x) = numerator(x)
    _ag_denominator(x) = denominator(x)
    _ag_qq_polynomial_ring(names) = polynomial_ring(QQ, String.(names))
    _ag_ideal(R, xs) = ideal(R, [R(x) for x in xs])
    _ag_complement_of_prime_ideal(I) = complement_of_prime_ideal(I; check=true)
    _ag_localization(R, S) = localization(R, S)
    _ag_zz_localization(p) = localization(ZZ, ZZ(p))
    _ag_in_ideal(x, I) = x in I
    function _ag_mpoly_from_terms(R, coefficients, exponents)
        variables = gens(R)
        result = zero(R)
        for k in eachindex(coefficients)
            term = R(coefficients[k])
            for j in eachindex(variables)
                term *= variables[j]^exponents[k][j]
            end
            result += term
        end
        return result
    end
    _ag_mpoly_coefficient_pairs(f) = [
        [BigInt(numerator(c)), BigInt(denominator(c))]
        for c in AbstractAlgebra.coefficients(f)
    ]
    _ag_mpoly_exponents(f) = [
        collect(e) for e in AbstractAlgebra.exponent_vectors(f)
    ]
end
''')
        self._generation = generation
        return self

    def call(self, function, *args):
        self.ensure_ready()
        return self._bridge.call(function, *args)


_OSCAR_BRIDGE = OscarBridgeDriver()


class OscarPolynomialPresentation(SageObject):
    """Exact transport between one Sage QQ-polynomial parent and one Oscar parent."""

    def __init__(self, sage_ring, oscar_ring, driver=None):
        if not isinstance(sage_ring, MPolynomialRing_base):
            raise TypeError(
                'the Sage source must be a multivariate polynomial ring'
            )
        if sage_ring.base_ring() is not QQ:
            raise NotImplementedError(
                'the exact Oscar polynomial transport currently supports coefficient field QQ'
            )
        if not isinstance(oscar_ring, JuliaHandle):
            raise TypeError(
                'the Oscar polynomial parent must be a persistent JuliaHandle'
            )
        self._sage_ring = sage_ring
        self._oscar_ring = oscar_ring
        self._driver = _OSCAR_BRIDGE if driver is None else driver

    def sage_ring(self):
        return self._sage_ring

    def oscar_ring(self):
        return self._oscar_ring

    def to_backend(self, polynomial):
        polynomial = self._sage_ring(polynomial)
        term_dict = polynomial.dict()
        exponents = [list(exponent) for exponent in term_dict]
        coefficients = [term_dict[exponent] for exponent in term_dict]
        return self._driver.call(
            '_ag_mpoly_from_terms',
            self._oscar_ring,
            coefficients,
            exponents,
        )

    def from_backend(self, polynomial_handle):
        coefficient_pairs = self._driver.call(
            '_ag_mpoly_coefficient_pairs',
            polynomial_handle,
        )
        exponents = self._driver.call(
            '_ag_mpoly_exponents',
            polynomial_handle,
        )
        if len(coefficient_pairs) != len(exponents):
            raise ArithmeticError(
                'Oscar returned inconsistent polynomial term data'
            )
        result = self._sage_ring.zero()
        variables = self._sage_ring.gens()
        for pair, exponent in zip(coefficient_pairs, exponents):
            numerator, denominator = map(ZZ, pair)
            coefficient = QQ(numerator) / QQ(denominator)
            term = self._sage_ring(coefficient)
            for variable, power in zip(variables, exponent):
                term *= variable ** ZZ(power)
            result += term
        return result


class OscarLocalRingElement(RingElement):
    def __init__(self, parent, backend_handle):
        if not isinstance(backend_handle, JuliaHandle):
            backend_handle = parent.driver().call(
                '_ag_coerce',
                parent.backend_handle(),
                backend_handle,
            )
        self._backend_handle = backend_handle
        RingElement.__init__(self, parent)

    def backend_handle(self):
        return self._backend_handle

    def _repr_(self):
        return self._backend_handle._display

    def _binary(self, function, other):
        other = self.parent()(other)
        return self.parent()._from_backend(
            self.parent().driver().call(
                function,
                self._backend_handle,
                other._backend_handle,
            )
        )

    def _add_(self, other):
        return self._binary('_ag_add', other)

    def _sub_(self, other):
        return self._binary('_ag_sub', other)

    def _mul_(self, other):
        return self._binary('_ag_mul', other)

    def _div_(self, other):
        other = self.parent()(other)
        if other.is_zero():
            raise ZeroDivisionError
        return self * ~other

    def _neg_(self):
        return self.parent()._from_backend(
            self.parent().driver().call(
                '_ag_neg',
                self._backend_handle,
            )
        )

    def __invert__(self):
        if not self.is_unit():
            raise ArithmeticError(
                'element is not a unit in the local ring'
            )
        return self.parent()._from_backend(
            self.parent().driver().call(
                '_ag_inv',
                self._backend_handle,
            )
        )

    def inverse_of_unit(self):
        return ~self

    def is_zero(self):
        return bool(
            self.parent().driver().call(
                '_ag_iszero',
                self._backend_handle,
            )
        )

    def is_one(self):
        return bool(
            self.parent().driver().call(
                '_ag_isone',
                self._backend_handle,
            )
        )

    def is_unit(self):
        return bool(
            self.parent().driver().call(
                '_ag_isunit',
                self._backend_handle,
            )
        )

    def numerator_backend(self):
        return self.parent().driver().call(
            '_ag_numerator',
            self._backend_handle,
        )

    def denominator_backend(self):
        return self.parent().driver().call(
            '_ag_denominator',
            self._backend_handle,
        )

    def _richcmp_(self, other, op):
        if op not in (op_EQ, op_NE):
            return NotImplemented
        try:
            other = self.parent()(other)
        except (TypeError, ValueError, NotImplementedError):
            return op == op_NE
        equal = bool(
            self.parent().driver().call(
                '_ag_eq',
                self._backend_handle,
                other._backend_handle,
            )
        )
        return equal if op == op_EQ else not equal


class OscarLocalRing(Parent):
    Element = OscarLocalRingElement

    def __init__(
        self,
        backend_handle,
        label,
        source_ring,
        source_to_backend,
        driver=None,
    ):
        if not isinstance(backend_handle, JuliaHandle):
            raise TypeError(
                'the local ring backend must be a persistent JuliaHandle'
            )
        self._backend_handle = backend_handle
        self._label = str(label)
        self._source_ring = source_ring
        self._source_to_backend = source_to_backend
        self._driver = _OSCAR_BRIDGE if driver is None else driver
        self._local_certificate = None
        Parent.__init__(self, category=LocalRings())

    def driver(self):
        return self._driver

    def backend_handle(self):
        return self._backend_handle

    def source_ring(self):
        return self._source_ring

    def _repr_(self):
        return self._label

    def _from_backend(self, value):
        if (
            isinstance(value, OscarLocalRingElement)
            and value.parent() is self
        ):
            return value
        if not isinstance(value, JuliaHandle):
            value = self._driver.call(
                '_ag_coerce',
                self._backend_handle,
                value,
            )
        return self.element_class(self, value)

    def _element_constructor_(self, value):
        if isinstance(value, OscarLocalRingElement):
            if value.parent() is self:
                return value
            value = value.backend_handle()
        try:
            if value.parent() is self._source_ring:
                return self._from_backend(
                    self._source_to_backend(value)
                )
        except AttributeError:
            pass
        return self._from_backend(value)

    def zero(self):
        return self._from_backend(
            self._driver.call(
                '_ag_coerce',
                self._backend_handle,
                ZZ.zero(),
            )
        )

    def one(self):
        return self._from_backend(
            self._driver.call(
                '_ag_coerce',
                self._backend_handle,
                ZZ.one(),
            )
        )

    def characteristic(self):
        return self._source_ring.characteristic()

    def is_exact(self):
        return True

    def is_local(self):
        return True

    def _set_local_certificate(self, certificate):
        if certificate.local_ring() is not self:
            raise ValueError(
                'the certificate belongs to another local ring'
            )
        if self._local_certificate is not None:
            raise RuntimeError(
                'the local-ring certificate is already installed'
            )
        self._local_certificate = certificate

    def local_axiom_certificate(self):
        if self._local_certificate is None:
            raise RuntimeError(
                'no local-ring certificate is installed'
            )
        return self._local_certificate

    def maximal_ideal(self):
        return self.local_axiom_certificate().maximal_ideal()

    def residue_field(self):
        return self.local_axiom_certificate().residue_field()

    def residue_morphism(self):
        return self.local_axiom_certificate().residue_morphism()

    def localization_morphism(self):
        return self.local_axiom_certificate().localization_morphism()


class CallableRingMorphism(RingHomomorphism):
    def __init__(self, domain, codomain, function, description):
        self._function = function
        self._description = str(description)
        RingHomomorphism.__init__(
            self,
            Hom(domain, codomain),
        )

    def _call_(self, element):
        result = self._function(self.domain()(element))
        try:
            if result.parent() is self.codomain():
                return result
        except AttributeError:
            pass
        return self.codomain()(result)

    def _repr_defn(self):
        return self._description


class CertifiedLocalIdeal(Ideal_generic):
    def __init__(
        self,
        ring,
        generators,
        membership,
        backend_handle=None,
    ):
        self._membership = membership
        self._backend_handle = backend_handle
        Ideal_generic.__init__(
            self,
            ring,
            generators,
        )

    def backend_handle(self):
        return self._backend_handle

    def __contains__(self, element):
        try:
            element = self.ring()(element)
        except (TypeError, ValueError, NotImplementedError):
            return False
        return bool(self._membership(element))

    def is_prime(self):
        return True

    def is_maximal(self):
        return True


class LocalRingCertificate(SageObject):
    """Theorem-backed certificate for R_p and its residue morphism."""

    def __init__(
        self,
        source_ring,
        prime_ideal,
        local_ring,
        localization_morphism,
        maximal_ideal,
        residue_field,
        residue_morphism,
        theorem,
        backend,
    ):
        if prime_ideal.ring() is not source_ring:
            raise ValueError(
                'the prime ideal belongs to another source ring'
            )
        if not prime_ideal.is_prime():
            raise ValueError(
                'the source ideal must be prime'
            )
        if local_ring not in LocalRings():
            raise TypeError(
                'the localized parent is not in LocalRings()'
            )
        if localization_morphism.domain() is not source_ring:
            raise ValueError(
                'the localization morphism has the wrong domain'
            )
        if localization_morphism.codomain() is not local_ring:
            raise ValueError(
                'the localization morphism has the wrong codomain'
            )
        if maximal_ideal.ring() is not local_ring:
            raise ValueError(
                'the maximal ideal belongs to another ring'
            )
        if residue_morphism.domain() is not local_ring:
            raise ValueError(
                'the residue morphism has the wrong domain'
            )
        if residue_morphism.codomain() is not residue_field:
            raise ValueError(
                'the residue morphism has the wrong codomain'
            )
        self._source_ring = source_ring
        self._prime_ideal = prime_ideal
        self._local_ring = local_ring
        self._localization_morphism = localization_morphism
        self._maximal_ideal = maximal_ideal
        self._residue_field = residue_field
        self._residue_morphism = residue_morphism
        self._theorem = str(theorem)
        self._backend = str(backend)

    def source_ring(self):
        return self._source_ring

    def prime_ideal(self):
        return self._prime_ideal

    def local_ring(self):
        return self._local_ring

    def localization_morphism(self):
        return self._localization_morphism

    def maximal_ideal(self):
        return self._maximal_ideal

    def residue_field(self):
        return self._residue_field

    def residue_morphism(self):
        return self._residue_morphism

    def theorem(self):
        return self._theorem

    def backend(self):
        return self._backend

    def proof_kind(self):
        return (
            'theorem-backed structural certificate '
            'with executed backend checks'
        )


class PrimeLocalizationUniversalProperty(
    LocalizationUniversalProperty
):
    def __init__(
        self,
        multiplicative_subset,
        localization_arrow,
        certificate,
    ):
        super().__init__(
            multiplicative_subset,
            localization_arrow,
        )
        self._certificate = certificate

    def local_axiom_certificate(self):
        return self._certificate

    def maximal_ideal(self):
        return self._certificate.maximal_ideal()

    def residue_field(self):
        return self._certificate.residue_field()

    def residue_morphism(self):
        return self._certificate.residue_morphism()

    def factor(self, base_map, certificate=None):
        if base_map is self.arrow():
            return self.localization_ring().Hom(
                self.localization_ring()
            ).identity()
        raise NotImplementedError(
            'Oscar constructs the prime localization, but executable '
            'factorization of an arbitrary external Sage ring morphism '
            'requires an exact target-map bridge and is not yet installed'
        )


print(
    'Installed LocalRings and Oscar-backed Sage parent, morphism, '
    'ideal, and certificate interfaces.'
)

Installed LocalRings and Oscar-backed Sage parent, morphism, ideal, and certificate interfaces.


In [ ]:
class OscarPrimeLocalizationBackend(SageObject):
    THEOREM = (
        'For a prime ideal p of a commutative ring R, the localization R_p '
        'is local with maximal ideal pR_p and residue field Frac(R/p).'
    )

    def __init__(self, driver=None):
        self._driver = _OSCAR_BRIDGE if driver is None else driver

    def driver(self):
        return self._driver

    def supports(self, ring, prime_ideal):
        if prime_ideal.ring() is not ring:
            return False
        if ring is ZZ:
            return len(prime_ideal.gens()) == 1
        return (
            isinstance(ring, MPolynomialRing_base)
            and ring.base_ring() is QQ
        )

    def localize(self, multiplicative_subset):
        if not isinstance(
            multiplicative_subset,
            ComplementOfPrimeIdeal,
        ):
            raise TypeError(
                'this backend requires the complement of a prime ideal'
            )
        prime_ideal = multiplicative_subset.prime_ideal()
        ring = multiplicative_subset.ring()
        if not self.supports(ring, prime_ideal):
            raise NotImplementedError(
                'the Oscar prime-localization backend currently supports ZZ '
                'and multivariate polynomial rings over QQ; the semantic '
                'localization operation remains defined for every commutative ring'
            )
        if ring is ZZ:
            return self._localize_integer_prime(
                multiplicative_subset
            )
        return self._localize_qq_polynomial_prime(
            multiplicative_subset
        )

    def _localize_integer_prime(self, multiplicative_subset):
        prime_ideal = multiplicative_subset.prime_ideal()
        generator = abs(ZZ(prime_ideal.gen()))
        if not generator.is_prime():
            raise ValueError(
                'the integer ideal must be generated by a prime integer'
            )
        local_handle = self._driver.call(
            '_ag_zz_localization',
            generator,
        )

        def source_to_backend(value):
            return self._driver.call(
                '_ag_coerce',
                local_handle,
                ZZ(value),
            )

        local_ring = OscarLocalRing(
            local_handle,
            f'Localization of Integer Ring at ({generator})',
            ZZ,
            source_to_backend,
            driver=self._driver,
        )
        localization_arrow = CallableRingMorphism(
            ZZ,
            local_ring,
            lambda value: local_ring._from_backend(
                source_to_backend(value)
            ),
            f'Canonical localization morphism ZZ -> ZZ_({generator})',
        )
        residue_field = GF(generator)

        def residue_function(element):
            numerator = ZZ(
                self._driver.call(
                    '_ag_numerator',
                    element.backend_handle(),
                )
            )
            denominator = ZZ(
                self._driver.call(
                    '_ag_denominator',
                    element.backend_handle(),
                )
            )
            denominator_image = residue_field(denominator)
            if denominator_image.is_zero():
                raise ArithmeticError(
                    'a localized denominator vanished in the residue field'
                )
            return residue_field(numerator) / denominator_image

        residue_morphism = CallableRingMorphism(
            local_ring,
            residue_field,
            residue_function,
            f'Residue morphism ZZ_({generator}) -> GF({generator})',
        )
        maximal_generator = localization_arrow(generator)
        maximal_ideal = CertifiedLocalIdeal(
            local_ring,
            (maximal_generator,),
            membership=lambda element: (
                residue_morphism(element).is_zero()
            ),
        )
        certificate = LocalRingCertificate(
            ZZ,
            prime_ideal,
            local_ring,
            localization_arrow,
            maximal_ideal,
            residue_field,
            residue_morphism,
            theorem=self.THEOREM,
            backend=(
                'Oscar LocalizedEuclideanRing '
                'with native Sage residue field'
            ),
        )
        local_ring._set_local_certificate(certificate)
        return PrimeLocalizationUniversalProperty(
            multiplicative_subset,
            localization_arrow,
            certificate,
        )

    def _localize_qq_polynomial_prime(
        self,
        multiplicative_subset,
    ):
        ring = multiplicative_subset.ring()
        prime_ideal = multiplicative_subset.prime_ideal()
        ring_tuple = self._driver.call(
            '_ag_qq_polynomial_ring',
            list(ring.variable_names()),
        )
        oscar_ring = self._driver.call(
            '_ag_first',
            ring_tuple,
        )
        presentation = OscarPolynomialPresentation(
            ring,
            oscar_ring,
            driver=self._driver,
        )
        oscar_prime = self._driver.call(
            '_ag_ideal',
            oscar_ring,
            [
                presentation.to_backend(generator)
                for generator in prime_ideal.gens()
            ],
        )
        oscar_complement = self._driver.call(
            '_ag_complement_of_prime_ideal',
            oscar_prime,
        )
        localization_tuple = self._driver.call(
            '_ag_localization',
            oscar_ring,
            oscar_complement,
        )
        local_handle = self._driver.call(
            '_ag_first',
            localization_tuple,
        )
        oscar_localization_map = self._driver.call(
            '_ag_second',
            localization_tuple,
        )

        def source_to_backend(value):
            return self._driver.call(
                '_ag_apply',
                oscar_localization_map,
                presentation.to_backend(value),
            )

        local_ring = OscarLocalRing(
            local_handle,
            f'Localization of {ring} at {prime_ideal}',
            ring,
            source_to_backend,
            driver=self._driver,
        )
        localization_arrow = CallableRingMorphism(
            ring,
            local_ring,
            lambda value: local_ring._from_backend(
                source_to_backend(value)
            ),
            (
                f'Canonical localization morphism '
                f'{ring} -> {ring}_{prime_ideal}'
            ),
        )
        quotient_names = tuple(
            f'{name}_bar'
            for name in ring.variable_names()
        )
        residue_ring = ring.quotient(
            prime_ideal,
            names=quotient_names,
        )
        residue_field = residue_ring.fraction_field()

        def residue_function(element):
            numerator_handle = self._driver.call(
                '_ag_numerator',
                element.backend_handle(),
            )
            denominator_handle = self._driver.call(
                '_ag_denominator',
                element.backend_handle(),
            )
            numerator = presentation.from_backend(
                numerator_handle
            )
            denominator = presentation.from_backend(
                denominator_handle
            )
            numerator_image = residue_field(
                residue_ring(numerator)
            )
            denominator_image = residue_field(
                residue_ring(denominator)
            )
            if denominator_image.is_zero():
                raise ArithmeticError(
                    'a localized denominator vanished modulo the prime ideal'
                )
            return numerator_image / denominator_image

        residue_morphism = CallableRingMorphism(
            local_ring,
            residue_field,
            residue_function,
            (
                f'Residue morphism {ring}_{prime_ideal} '
                f'-> Frac({ring}/{prime_ideal})'
            ),
        )
        localized_prime_generators = tuple(
            localization_arrow(generator)
            for generator in prime_ideal.gens()
        )
        oscar_maximal_ideal = self._driver.call(
            '_ag_ideal',
            local_handle,
            [
                generator.backend_handle()
                for generator in localized_prime_generators
            ],
        )
        maximal_ideal = CertifiedLocalIdeal(
            local_ring,
            localized_prime_generators,
            membership=lambda element: self._driver.call(
                '_ag_in_ideal',
                element.backend_handle(),
                oscar_maximal_ideal,
            ),
            backend_handle=oscar_maximal_ideal,
        )
        certificate = LocalRingCertificate(
            ring,
            prime_ideal,
            local_ring,
            localization_arrow,
            maximal_ideal,
            residue_field,
            residue_morphism,
            theorem=self.THEOREM,
            backend=(
                'Oscar MPolyLocRing with exact Sage '
                'finite-presentation transport'
            ),
        )
        local_ring._set_local_certificate(certificate)
        return PrimeLocalizationUniversalProperty(
            multiplicative_subset,
            localization_arrow,
            certificate,
        )


_OSCAR_PRIME_LOCALIZATION_BACKEND = (
    OscarPrimeLocalizationBackend()
)

print(
    'Installed Oscar prime-localization dispatch for ZZ and '
    'multivariate polynomial rings over QQ.'
)

In [ ]:
R_local_regression = PolynomialRing(
    QQ,
    names=('x_local_regression', 'y_local_regression'),
)
x_local_regression, y_local_regression = (
    R_local_regression.gens()
)
base_certificate_local_regression = (
    PolynomialGeneratorEqualityCertificate(
        R_local_regression
    )
)

multiplicative_subset_regression = (
    R_local_regression.multiplicative_subset(
        (
            x_local_regression,
            y_local_regression,
        )
    )
)
localization_regression = (
    R_local_regression.localization_morphism(
        multiplicative_subset_regression
    )
)
ell_local_regression = (
    localization_regression.arrow()
)
L_local_regression = (
    localization_regression.localization_ring()
)
factor_local_regression = (
    localization_regression.factor(
        ell_local_regression
    )
)
localization_equality_regression = (
    LocalizationEqualityCertificate(
        ell_local_regression,
        base_certificate_local_regression,
    )
)
assert base_certificate_local_regression.verify(
    factor_local_regression
    * ell_local_regression,
    ell_local_regression,
)
assert localization_equality_regression.verify(
    factor_local_regression,
    L_local_regression.Hom(
        L_local_regression
    ).identity(),
)

R_quotient_regression = PolynomialRing(
    QQ,
    names=('z_quotient_regression',),
)
z_quotient_regression = (
    R_quotient_regression.gen()
)
I_quotient_regression = (
    R_quotient_regression.ideal(
        z_quotient_regression**2
    )
)
quotient_regression = (
    R_quotient_regression.quotient_morphism(
        I_quotient_regression,
        names=('zbar_quotient_regression',),
    )
)
q_quotient_regression = (
    quotient_regression.arrow()
)
evaluation_regression = (
    R_quotient_regression.hom(
        (QQ.zero(),),
        QQ,
    )
)
factor_quotient_regression = (
    quotient_regression.factor(
        evaluation_regression
    )
)
base_certificate_quotient_regression = (
    PolynomialGeneratorEqualityCertificate(
        R_quotient_regression
    )
)
quotient_equality_regression = (
    QuotientEqualityCertificate(
        q_quotient_regression,
        base_certificate_quotient_regression,
    )
)
assert base_certificate_quotient_regression.verify(
    factor_quotient_regression
    * q_quotient_regression,
    evaluation_regression,
)
assert quotient_equality_regression.verify(
    factor_quotient_regression,
    factor_quotient_regression,
)

R_fraction_regression = PolynomialRing(
    QQ,
    names=('t_fraction_regression',),
)
t_fraction_regression = (
    R_fraction_regression.gen()
)
fraction_regression = (
    R_fraction_regression.fraction_field_morphism()
)
ell_fraction_regression = (
    fraction_regression.arrow()
)
K_fraction_regression = (
    fraction_regression.localization_ring()
)
injective_fraction_regression = (
    InjectiveMapToFieldCertificate(
        ell_fraction_regression,
        theorem=(
            'The canonical morphism from an integral domain '
            'to its fraction field is injective.'
        ),
    )
)
factor_fraction_regression = (
    fraction_regression.factor(
        ell_fraction_regression,
        injective_fraction_regression,
    )
)
base_certificate_fraction_regression = (
    PolynomialGeneratorEqualityCertificate(
        R_fraction_regression
    )
)
fraction_equality_regression = (
    LocalizationEqualityCertificate(
        ell_fraction_regression,
        base_certificate_fraction_regression,
    )
)
assert base_certificate_fraction_regression.verify(
    factor_fraction_regression
    * ell_fraction_regression,
    ell_fraction_regression,
)
assert fraction_equality_regression.verify(
    factor_fraction_regression,
    K_fraction_regression.Hom(
        K_fraction_regression
    ).identity(),
)

integer_prime_localization_regression = (
    ZZ.localization_at_prime(
        ZZ.ideal(5)
    )
)
Z5_regression = (
    integer_prime_localization_regression.localization_ring()
)
ell_Z5_regression = (
    integer_prime_localization_regression.arrow()
)
m_Z5_regression = (
    integer_prime_localization_regression.maximal_ideal()
)
kappa_Z5_regression = (
    integer_prime_localization_regression.residue_field()
)
rho_Z5_regression = (
    integer_prime_localization_regression.residue_morphism()
)
assert Z5_regression in LocalRings()
assert Z5_regression.is_local()
assert ell_Z5_regression(2).is_unit()
assert not ell_Z5_regression(5).is_unit()
assert ell_Z5_regression(5) in m_Z5_regression
assert ell_Z5_regression(2) not in m_Z5_regression
assert rho_Z5_regression(
    ell_Z5_regression(7)
) == kappa_Z5_regression(2)
assert rho_Z5_regression(
    ell_Z5_regression(5)
).is_zero()
assert integer_prime_localization_regression.factor(
    ell_Z5_regression
).is_identity()

R_prime_regression = PolynomialRing(
    QQ,
    names=(
        'x_prime_regression',
        'y_prime_regression',
    ),
)
x_prime_regression, y_prime_regression = (
    R_prime_regression.gens()
)
prime_ideal_regression = (
    R_prime_regression.ideal(
        x_prime_regression
    )
)
polynomial_prime_localization_regression = (
    R_prime_regression.localization_at_prime(
        prime_ideal_regression
    )
)
Rpx_regression = (
    polynomial_prime_localization_regression.localization_ring()
)
ell_Rpx_regression = (
    polynomial_prime_localization_regression.arrow()
)
mpx_regression = (
    polynomial_prime_localization_regression.maximal_ideal()
)
kappa_Rpx_regression = (
    polynomial_prime_localization_regression.residue_field()
)
rho_Rpx_regression = (
    polynomial_prime_localization_regression.residue_morphism()
)
assert Rpx_regression in LocalRings()
assert Rpx_regression.is_local()
assert not ell_Rpx_regression(
    x_prime_regression
).is_unit()
assert ell_Rpx_regression(
    y_prime_regression
).is_unit()
assert ell_Rpx_regression(
    x_prime_regression
) in mpx_regression
assert ell_Rpx_regression(
    y_prime_regression
) not in mpx_regression
assert rho_Rpx_regression(
    ell_Rpx_regression(x_prime_regression)
).is_zero()
assert not rho_Rpx_regression(
    ell_Rpx_regression(y_prime_regression)
).is_zero()
assert (
    rho_Rpx_regression(
        ~ell_Rpx_regression(y_prime_regression)
    )
    * rho_Rpx_regression(
        ell_Rpx_regression(y_prime_regression)
    )
    == 1
)
prime_fraction_regression = (
    ell_Rpx_regression(x_prime_regression + 1)
    / ell_Rpx_regression(y_prime_regression)
)
assert rho_Rpx_regression(
    prime_fraction_regression
) == (
    1
    / rho_Rpx_regression(
        ell_Rpx_regression(y_prime_regression)
    )
)
assert polynomial_prime_localization_regression.factor(
    ell_Rpx_regression
).is_identity()

print('Local universal-arrow regressions passed.')
print('localization ring =', L_local_regression)
print('quotient ring =', quotient_regression.quotient_ring())
print('fraction field =', K_fraction_regression)
print('integer prime local ring =', Z5_regression)
print('integer maximal ideal =', m_Z5_regression)
print('integer residue field =', kappa_Z5_regression)
print('polynomial prime local ring =', Rpx_regression)
print('polynomial maximal ideal =', mpx_regression)
print('polynomial residue field =', kappa_Rpx_regression)
print(
    'residue of (x+1)/y =',
    rho_Rpx_regression(prime_fraction_regression),
)